In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Creating subagents

In [3]:
from langchain.tools import tool
from typing import Literal

@tool
def calculate_percentage(total_marks: int, obtained_marks: float) -> float:
    """Calculate the percentage from the given total and obtained marks"""
    return (obtained_marks/total_marks) * 100

@tool
def assign_grade(percentage: float) -> str:
    """Assign grade Pass or Fail depending on the percentage"""

    if percentage < float(50):
        return "Fail"
    else:
        return "Pass"

In [4]:
from langchain.agents import create_agent

# create subagents

percentage_calculator_agent = create_agent(
    model='gpt-5-nano',
    tools=[calculate_percentage]
)

grade_assigner_agent = create_agent(
    model='gpt-5-nano',
    tools=[assign_grade]
)

## Calling subagents

In [5]:
from langchain.messages import HumanMessage

@tool
def percentage_calculator(total_marks: int, obtained_marks: float) -> float:
    """Call subagent 1 in order to calculate the square root of a number"""
    response = percentage_calculator_agent.invoke({"messages": [HumanMessage(content=f"Calculate percentage -> Total Marks: {total_marks}. Obtained Marks: {obtained_marks}")]})
    return response["messages"][-1].content

@tool
def grade_assigner(percentage: float) -> str:
    """Call subagent 2 in order to calculate the square of a number"""
    response = grade_assigner_agent.invoke({"messages": [HumanMessage(content=f"Assign grade for this percentage: {percentage}")]})
    return response["messages"][-1].content

## Creating the main agent

main_agent = create_agent(
    model='gpt-5-nano',
    tools=[percentage_calculator, grade_assigner],
    system_prompt="You are a helpful assistant who can call subagents to calculate the percentage from given total marks and obtained marks and then assign grade. by must calling sub agents, First calculate percentage, then assign grade.")

## Test

In [6]:
question = "what will be my grade if my obtained marks are 250 out of 400?"

response = main_agent.invoke({"messages": [HumanMessage(content=question)]})

In [7]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='what will be my grade if my obtained marks are 250 out of 400?', additional_kwargs={}, response_metadata={}, id='df05515d-33f9-4eb9-baa3-f4448496573a'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 224, 'prompt_tokens': 231, 'total_tokens': 455, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 192, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-E9wgv1FIYN3RvbO2fd34aUetdhT5h', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fd83c-6705-7ce0-ac46-5c1765b991ef-0', tool_calls=[{'name': 'percentage_calculator', 'args': {'total_marks': 400, 'obtained_marks': 250}, 'id': 'call_4wAO89ieeTiTXdRgL2oqc6A0', 'type': 'tool_c

In [8]:
pprint(response["messages"][-1].content)

'Your percentage is 62.5%, and your grade is: Pass.'
